# RetinaGuard — ALL-IN-ONE unattended training + auto-publish (Colab)

Runs the entire pipeline on its own (overnight), saves everything to **Google Drive** live, then at the end **commits the proof artifacts to GitHub** and **publishes the trained weights as a GitHub Release**.

## Do these ONCE before running
1. **Runtime → Change runtime type → GPU (T4)**, then Save.
2. Put your **`kaggle.json`** in the ROOT of your Google Drive (My Drive). (Kaggle → Account → Create New API Token.)
3. **Join the APTOS 2019 competition** once (accept rules): https://www.kaggle.com/c/aptos2019-blindness-detection/rules
4. **Add a GitHub token as a Colab Secret** so the notebook can push + release:
   - GitHub → Settings → Developer settings → **Fine-grained personal access tokens** → Generate. Repository access = only `RetinaGuard`. Permissions → **Contents: Read and write**.
   - In Colab, click the **🔑 (Secrets)** icon in the left sidebar → **+ Add new secret** → Name it exactly **`GITHUB_TOKEN`** → paste the token → enable **Notebook access**.

## Then
Run **Cell 1** (authorize Google Drive — ~20 sec, the only manual step). Then **Runtime → Run all** and walk away.

Results land in `My Drive/RetinaGuard_outputs/`, the proof artifacts get committed to `docs/results/`, and the `.pth` is attached to a new GitHub Release.

### To keep it alive overnight
- Keep the browser tab open; **disable sleep** on your computer (plug it in).
- Free Colab caps sessions (~12h) and may disconnect on idle. Checkpoints are written to Drive live, so a disconnect still leaves you the best model trained so far (re-run the eval + publish cells against it later).

## Cell 1 — Mount Google Drive (only manual step)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUT = '/content/drive/MyDrive/RetinaGuard_outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
assert os.path.exists('/content/drive/MyDrive/kaggle.json'), \
    'Put kaggle.json in the ROOT of My Drive before running.'
print('Drive mounted. Outputs ->', DRIVE_OUT)

## Cell 2 — Config knobs (safe defaults for unattended T4)
512px EfficientNet-B4 can OOM on a free T4. Defaults use 380px (B4's native resolution) for a reliable overnight run. For original 512px set `IMG = 512` and `BS = 8`.

In [ ]:
IMG    = 380   # image size (px). 380 = safe/fast on T4. 512 = original (higher VRAM).
BS     = 16    # batch size. Lower to 8 if you see CUDA out of memory.
EPOCHS = 60    # max epochs (early stopping usually ends sooner).
GITHUB_REPO = 'Gaurav-0704/RetinaGuard'   # owner/repo to push proofs + release to

## Cell 3 — One-shot setup: clone, install, get data, patch config, link weights to Drive

In [ ]:
import os, subprocess, shutil, re
%cd /content
if os.path.isdir('RetinaGuard'): shutil.rmtree('RetinaGuard')
!git clone https://github.com/Gaurav-0704/RetinaGuard.git
%cd /content/RetinaGuard

# Deps (Colab already has torch/sklearn/pandas/matplotlib)
!pip -q install opencv-python-headless kaggle >/dev/null 2>&1

# Kaggle auth from Drive
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download APTOS 2019 (train images + labels)
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    !kaggle competitions download -c aptos2019-blindness-detection -f train.csv -p data
    !cd data && (unzip -o -q train.csv.zip 2>/dev/null || true)
if not os.path.isdir('data/train_images'):
    !kaggle competitions download -c aptos2019-blindness-detection -f train_images.zip -p data
    !cd data && unzip -o -q train_images.zip -d train_images
import pandas as pd
_df = pd.read_csv('data/train.csv'); print('images csv rows:', len(_df))
print(_df['diagnosis'].value_counts().sort_index())

# Patch hyperparameters from Cell 2
cfg = open('ml/config.py').read()
cfg = re.sub(r'^IMAGE_SIZE.*$', f'IMAGE_SIZE        = {IMG}', cfg, flags=re.M)
cfg = re.sub(r'^BATCH_SIZE.*$',  f'BATCH_SIZE        = {BS}',  cfg, flags=re.M)
cfg = re.sub(r'^NUM_EPOCHS.*$',  f'NUM_EPOCHS        = {EPOCHS}', cfg, flags=re.M)
open('ml/config.py','w').write(cfg)
print(f'Config patched -> IMG={IMG} BS={BS} EPOCHS={EPOCHS}')

# Make backend/weights a symlink into Drive so checkpoints persist LIVE
drive_weights = '/content/drive/MyDrive/RetinaGuard_outputs/weights'
os.makedirs(drive_weights, exist_ok=True)
if os.path.islink('backend/weights'):
    os.unlink('backend/weights')
elif os.path.isdir('backend/weights'):
    shutil.rmtree('backend/weights')
os.symlink(drive_weights, 'backend/weights')
print('backend/weights -> Drive:', os.readlink('backend/weights'))

# Verify code parses before the long run
!python -c "import ast; ast.parse(open('ml/train.py').read()); ast.parse(open('ml/evaluate.py').read()); print('train+evaluate parse OK')"
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Cell 4 — Smoke test (2 epochs, ~few min). Confirms the full run will work.

In [ ]:
!cp ml/config.py /tmp/config_full.py
!sed -i 's/^NUM_EPOCHS.*/NUM_EPOCHS        = 2/' ml/config.py
!python -m ml.train --images_dir data/train_images --csv_path data/train.csv
!python -m ml.evaluate --images_dir data/train_images --csv_path backend/weights/test_split.csv
!cp /tmp/config_full.py ml/config.py   # restore full epochs
print('SMOKE TEST OK — a classification report + metrics above means you are good to go.')

## Cell 5 — FULL training + evaluation + save to Drive (the long one; runs unattended)

In [ ]:
import subprocess, datetime, shutil, os
log = '/content/drive/MyDrive/RetinaGuard_outputs/train_log.txt'
print('Started:', datetime.datetime.now())

# Train (stream log to Drive so you can check progress from your phone)
with open(log, 'w') as f:
    p = subprocess.Popen(['python','-m','ml.train','--images_dir','data/train_images',
                          '--csv_path','data/train.csv'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end=''); f.write(line); f.flush()
    p.wait()
assert p.returncode == 0, 'Training failed — check the log above.'

# Evaluate on the held-out test split
!python -m ml.evaluate --images_dir data/train_images --csv_path backend/weights/test_split.csv

# Copy results into Drive (weights are already there via symlink)
if os.path.isdir('docs/results'):
    shutil.copytree('docs/results', '/content/drive/MyDrive/RetinaGuard_outputs/results', dirs_exist_ok=True)
print('FINISHED:', datetime.datetime.now())
print('Saved in My Drive/RetinaGuard_outputs/  (weights/, results/, train_log.txt)')

## Cell 6 — Show the results

In [ ]:
print(open('docs/results/report.md').read())
from IPython.display import Image, display
for p in ['docs/results/confusion_matrix.png','docs/results/confusion_matrix_normalized.png','docs/results/roc_curves.png']:
    display(Image(p))

## Cell 7 — Commit proof artifacts to GitHub + publish weights as a Release
Commits the small, verifiable proofs (metrics.json, report.md, confusion matrix + ROC PNGs, history.json, test_split.csv) to `docs/results/`, then creates a GitHub Release and uploads `retinoguard_best.pth` as an asset. Requires the `GITHUB_TOKEN` secret (see top of notebook). No large files ever enter git history.

In [ ]:
import os, json, shutil, datetime, requests
from google.colab import userdata

TOKEN = userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add a Colab secret named GITHUB_TOKEN (fine-grained PAT, Contents: read/write).'

# 1) Stage proof artifacts inside the repo (small text/image only — never the weights)
os.makedirs('docs/results', exist_ok=True)
for src in ['backend/weights/history.json', 'backend/weights/test_split.csv']:
    if os.path.exists(src):
        shutil.copy(src, 'docs/results/')
metrics = json.load(open('docs/results/metrics.json'))
print('Metrics to publish:', metrics['overall'])

# 2) Commit + push the proofs (token-authenticated remote)
!git config user.email "gauravsinght007@gmail.com"
!git config user.name "Gaurav-0704"
!git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git
!git add docs/results docs/RESULTS.md
!git commit -m "Add training proof artifacts (metrics, confusion matrix, ROC, history, held-out split)" || echo "nothing to commit"
!git push origin HEAD:main

# 3) Create a GitHub Release and upload the .pth as an asset
H = {'Authorization': f'token {TOKEN}', 'Accept': 'application/vnd.github+json'}
tag = 'model-v1-' + datetime.datetime.now().strftime('%Y%m%d-%H%M')
o = metrics['overall']
body = (f"RetinaGuard EfficientNet-B4 trained on APTOS 2019 (held-out test set: "
        f"{metrics['n_test_samples']} images).\n\n"
        f"- Accuracy: {o['accuracy']}\n"
        f"- Quadratic Weighted Kappa: {o['quadratic_weighted_kappa']}\n"
        f"- Macro F1: {o['f1_macro']}\n"
        f"- Macro AUC (OvR): {o['macro_auc_ovr']}\n\n"
        f"Full per-class metrics, confusion matrix and ROC curves are in docs/results/.")
r = requests.post(f'https://api.github.com/repos/{GITHUB_REPO}/releases', headers=H,
                  json={'tag_name': tag, 'name': f'Trained model {tag}', 'body': body})
r.raise_for_status()
rel = r.json(); upload_url = rel['upload_url'].split('{')[0]
print('Release created:', rel['html_url'])

pth = os.path.realpath('backend/weights/retinoguard_best.pth')
assert os.path.exists(pth), 'No trained checkpoint found to upload.'
with open(pth, 'rb') as fh:
    up = requests.post(upload_url + '?name=retinoguard_best.pth',
                       headers={**H, 'Content-Type': 'application/octet-stream'}, data=fh)
up.raise_for_status()
print('Weights uploaded:', up.json()['browser_download_url'])
print('\nDONE. Proofs committed to docs/results/, weights attached to the Release above.')